# 국민여행조사 통합본 전처리

## 분석 목적과 범위

- `data/preprocess/national_travel_survey_2023_2025.csv`(2023~2025년
  국내여행 3개년 통합본, UTF-8-SIG)를 분류 모델링(Dummy·Elastic
  Net·의사결정나무·랜덤 포레스트·CatBoost·Soft voting·Stacking)과
  클러스터링(K-means·K-medoids·Gower 거리·오토인코더 임베딩)에 투입할
  변수셋으로 정리해 저장한다.
- 분석 단위는 응답자·조사회차이며 기본 키는 `YEAR + ID`다.
- 전처리 4단계 중 1단계는 제외한다.
    1. `D_TRA{k}_CHECK == "Y"`인 여행 슬롯(A계열 본설문 응답 대상
       여행)의 값을 `D_TRA_*` 열로 옮기고 `D_TRA1_*`~`D_TRA6_*` 원본
       슬롯 열을 모두 삭제한다.
    2. 중복 질문 열(`DQ1, DQ1A, DQ2, DQ2A, DQ3, TDQ3_5, DQ6A, DQ6B`)과
       불필요한 열(`MON_EXP_1~5, DQ4, DQ5, DQ5AA_1, DQ5AA_2, DQ5AB,
       DQ5AD, DQ5B`)을 삭제한다.
    3. 손자녀 수 열(`DQ3A_1~4`)을 합쳐 자녀 수 열 `DQ3A` 하나로
       관리한다.
    4. 3순위까지의 순위형 변수 5그룹(`A3_1~3, A5_1~3, A5A_1~3,
       ZQ1_1~3, D_TRA_B7_1~3`)을 카테고리 코드별 가중 다중응답 열
       `{그룹}_RANKW_{코드}`로 변환한다. 1순위 3점, 2순위 2점, 3순위
       1점, 미선택 0점을 부여하고 원본 순위 열은 유지한다.
- `D_TRA{k}_CHECK`가 모든 슬롯에서 비어 있는 응답자는 A계열 본설문
  응답 대상 여행이 없다는 뜻이며(`use-national-travel-survey` 스킬의
  CHECK 정의), 이는 여행 자체가 없다는 의미가 아니다. 통합 후
  `D_TRA_*` 열은 이 응답자에서 결측으로 남는다.
- 코드북(`national_travel_survey_2023_2025_codebook.csv`)에서 숫자로
  기록된 변수도 `column_type`이 `nominal`이면 범주형으로 관리해야
  한다. 데이터 로드 시 전체 열을 문자열(`dtype=str`)로 읽어 자동
  숫자 추론을 막고, 실제로 산술이 필요한 열만 셀 단위로 명시적으로
  숫자형으로 변환한다.

## 환경, 공통 경로와 재현성 설정

In [ ]:
from pathlib import Path
import re
import sys


current_directory = Path.cwd().resolve()
project_root = next(
    (
        candidate
        for candidate in (current_directory, *current_directory.parents)
        if (candidate / "src" / "path.py").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "프로젝트 루트를 찾을 수 없습니다. 프로젝트 내부에서 실행하세요."
    )
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

from src.path import NATIONAL_TRAVEL_SURVEY_PREPROCESSED_DATA_DIR  # noqa: E402


INPUT_DATA_PATH = NATIONAL_TRAVEL_SURVEY_PREPROCESSED_DATA_DIR / "national_travel_survey_2023_2025.csv"
INPUT_CODEBOOK_PATH = (
    NATIONAL_TRAVEL_SURVEY_PREPROCESSED_DATA_DIR / "national_travel_survey_2023_2025_codebook.csv"
)
OUTPUT_DATA_PATH = (
    NATIONAL_TRAVEL_SURVEY_PREPROCESSED_DATA_DIR / "national_travel_survey_2023_2025_all_tour.csv"
)
OUTPUT_CODEBOOK_PATH = (
    NATIONAL_TRAVEL_SURVEY_PREPROCESSED_DATA_DIR
    / "national_travel_survey_2023_2025_all_tour_codebook.csv"
)
TRIP_SLOTS = range(1, 7)

## 데이터 로드

전체 열을 문자열로 읽어 명목형 코드가 숫자로 자동 추론되지 않도록
한다. 결측은 CSV의 빈 칸 그대로 `NaN`으로 유지된다.

In [13]:
codebook = pd.read_csv(INPUT_CODEBOOK_PATH, encoding="utf-8-sig")
data = pd.read_csv(INPUT_DATA_PATH, encoding="utf-8-sig", dtype=str)

assert set(codebook["column_name"]) == set(data.columns), (
    "코드북과 통합본의 열 구성이 다릅니다."
)
codebook_by_name = codebook.set_index("column_name", drop=False)

print(f"통합본 크기: {data.shape[0]:,}행 × {data.shape[1]:,}열")
print(codebook["column_type"].value_counts())

통합본 크기: 156,050행 × 1,405열
column_type
nominal    960
scale      445
Name: count, dtype: int64


## 데이터 품질과 분석 단위 확인

In [14]:
assert data["ID"].notna().all(), "ID 결측이 있습니다."
assert not data[["YEAR", "ID"]].duplicated().any(), (
    "YEAR + ID 키가 중복됩니다."
)

weight = pd.to_numeric(data["WT_DOM"], errors="coerce")
assert weight.notna().all(), "WT_DOM 결측이 있습니다."
assert weight.gt(0).all(), "WT_DOM에 0 이하 값이 있습니다."

check_columns = [f"D_TRA{slot}_CHECK" for slot in TRIP_SLOTS]
y_slot_count = data[check_columns].eq("Y").sum(axis=1)
print("응답자별 CHECK == 'Y' 슬롯 개수 분포(0 또는 1이어야 함):")
print(y_slot_count.value_counts().sort_index())
assert y_slot_count.le(1).all(), (
    "한 응답자에서 CHECK == 'Y'인 슬롯이 2개 이상입니다."
)

응답자별 CHECK == 'Y' 슬롯 개수 분포(0 또는 1이어야 함):
0    83917
1    72133
Name: count, dtype: int64


## 전처리 1) 여행 슬롯 통합(D_TRA)

`D_TRA{k}_CHECK == "Y"`인 슬롯이 A계열 본설문 응답 대상 여행이다.
응답자당 `Y` 슬롯은 최대 1개이므로, 슬롯별 열 전체를 접두사만 다른
`D_TRA_*` 열로 모으고 `D_TRA1_*`~`D_TRA6_*` 원본 슬롯 열은 삭제한다.
`Y` 슬롯이 없는 응답자는 `D_TRA_*`가 전부 결측이 된다.

In [ ]:
# original_columns = data.columns.tolist()
# d_tra_slot_pattern = re.compile(r"^D_TRA[1-6]_")

# first_slot_prefix = "D_TRA1_"
# suffixes = [
#     column[len(first_slot_prefix):]
#     for column in original_columns
#     if column.startswith(first_slot_prefix)
# ]
# for slot in TRIP_SLOTS:
#     slot_prefix = f"D_TRA{slot}_"
#     slot_suffixes = [
#         column[len(slot_prefix):]
#         for column in original_columns
#         if column.startswith(slot_prefix)
#     ]
#     assert slot_suffixes == suffixes, (
#         f"D_TRA{slot}_* 열 구성이 D_TRA1_*와 다릅니다."
#     )
# print(f"여행 슬롯당 항목 수: {len(suffixes)}")

여행 슬롯당 항목 수: 169


In [ ]:
# check_matrix = data[[f"D_TRA{slot}_CHECK" for slot in TRIP_SLOTS]].eq("Y").to_numpy()
# has_target_trip = check_matrix.any(axis=1)
# target_slot_index = check_matrix.argmax(axis=1)
# row_index = np.arange(len(data))

# integrated_trip = {}
# for suffix in suffixes:
#     slot_values = data[
#         [f"D_TRA{slot}_{suffix}" for slot in TRIP_SLOTS]
#     ].to_numpy()
#     selected = slot_values[row_index, target_slot_index]
#     integrated_trip[f"D_TRA_{suffix}"] = np.where(
#         has_target_trip, selected, np.nan
#     )

# data = data.assign(**integrated_trip)
# data = data.drop(
#     columns=[c for c in original_columns if d_tra_slot_pattern.match(c)]
# )

# d_tra_start = original_columns.index("D_TRA1_CHECK")
# before_d_tra = original_columns[:d_tra_start]
# after_d_tra = [
#     c for c in original_columns
#     if not d_tra_slot_pattern.match(c) and c not in before_d_tra
# ]
# new_trip_columns = [f"D_TRA_{suffix}" for suffix in suffixes]
# data = data[before_d_tra + new_trip_columns + after_d_tra]

# print(f"D_TRA_CASE 비결측 응답자 수: {data['D_TRA_CASE'].notna().sum():,}")
# assert data["D_TRA_CASE"].notna().sum() == has_target_trip.sum()

D_TRA_CASE 비결측 응답자 수: 72,133


## 전처리 2) 중복·불필요 열 삭제

같은 내용을 다시 묻는 열(예: `DQ1`↔`BEDU`, `DQ3`↔`BMAR`, `DQ6A/DQ6B`↔
`BINC1/BINC2`)과 이번 분석에 쓰지 않는 근로 관련 열을 삭제한다.

In [15]:
duplicate_columns = [
    "DQ1", "DQ1A", "DQ2", "DQ2A", "DQ3", "TDQ3_5", "DQ6A", "DQ6B",
]
unneeded_columns = [
    "MON_EXP_1", "MON_EXP_2", "MON_EXP_3", "MON_EXP_4", "MON_EXP_5",
    "DQ4", "DQ5", "DQ5AA_1", "DQ5AA_2", "DQ5AB", "DQ5AD", "DQ5B", "A6C",
]
columns_to_drop = duplicate_columns + unneeded_columns

assert set(columns_to_drop) <= set(data.columns), (
    "삭제 대상 열 중 존재하지 않는 열이 있습니다."
)
data = data.drop(columns=columns_to_drop)
print(f"열 삭제 후 크기: {data.shape[0]:,}행 × {data.shape[1]:,}열")

열 삭제 후 크기: 156,050행 × 1,384열


## 전처리 3) 손자녀 수 열 통합(DQ3A)

`DQ3A_1`은 "동거 손자/자녀 없음" 체크(Y/결측)이고 `DQ3A_2~4`는
연령대별 동거 손자/자녀 수(명)다. `DQ3A_1 == "Y"`인 행은 `DQ3A_2~4`가
모두 0임을 확인한 뒤, `DQ3A_2~4`를 숫자로 변환해 합산한 `DQ3A`(자녀
수)를 만든다. 세 열이 모두 결측이면(`DQ3`가 미혼이라 구조적으로
비해당인 경우) `DQ3A`도 결측으로 남긴다.

In [16]:
child_count_columns = ["DQ3A_2", "DQ3A_3", "DQ3A_4"]
child_counts = data[child_count_columns].apply(pd.to_numeric)

no_child_mask = data["DQ3A_1"].eq("Y")
assert child_counts.loc[no_child_mask].eq(0).all().all(), (
    "DQ3A_1 == 'Y'인 행에 0이 아닌 손자녀 수가 있습니다."
)

dq3a = child_counts.sum(axis=1, min_count=1)

original_columns_before_dq3a = data.columns.tolist()
dq3a_columns = ["DQ3A_1", "DQ3A_2", "DQ3A_3", "DQ3A_4"]
insert_at = original_columns_before_dq3a.index("DQ3A_1")
remaining_columns = [
    c for c in original_columns_before_dq3a if c not in dq3a_columns
]

data = data.drop(columns=dq3a_columns)
data["DQ3A"] = dq3a
data = data[
    remaining_columns[:insert_at] + ["DQ3A"] + remaining_columns[insert_at:]
]

print("DQ3A(자녀 수) 분포:")
print(data["DQ3A"].value_counts(dropna=False).sort_index())

DQ3A(자녀 수) 분포:
DQ3A
0.0    66375
1.0    24626
2.0    20572
3.0     1578
4.0      180
5.0       23
NaN    42696
Name: count, dtype: int64


## 전처리 4) 순위 가중 다중응답 변수(RANKW)

3순위까지 응답하는 순위형 변수 5그룹(`A3_1~3` 여행지역 선택 이유,
`A5_1~3` 여행정보 출처, `A5A_1~3` 인터넷/웹 세부 출처, `ZQ1_1~3`
미여행 이유, `D_TRA_B7_1~3` 1차 여행 주요 이동수단)을 카테고리
코드별 가중 다중응답 열 `{그룹}_RANKW_{코드}`로 변환한다. 1순위는
3점, 2순위는 2점, 3순위는 1점, 미선택은 0점을 부여한다. 같은
카테고리가 여러 순위에 중복 응답된 경우(`A5A`, `ZQ1`에서 확인됨)는
`np.maximum`으로 최고 순위 점수만 인정한다. 그룹의 1순위 문항 자체가
결측인 응답자는 해당 문항에 응답할 대상이 아니었다는 구조적
비해당이므로 그룹의 모든 가중 열을 결측으로 남긴다(0으로 채우지
않는다). 카테고리 코드는 코드북 전체 라벨이 아니라 데이터에 실제로
관측된 값만 사용한다. 원본 순위 열(`A3_1~3` 등)은 삭제하지 않고
그대로 둔다.

In [18]:
import json  # noqa: E402


RANK_VARIABLE_GROUPS = {
    "A3": [f"A3_{rank}" for rank in (1, 2, 3)],
    "A5": [f"A5_{rank}" for rank in (1, 2, 3)],
    "A5A": [f"A5A_{rank}" for rank in (1, 2, 3)],
    "ZQ1": [f"ZQ1_{rank}" for rank in (1, 2, 3)],
    "D_TRA1_B7": [f"D_TRA1_B7_{rank}" for rank in (1, 2, 3)],
    "D_TRA2_B7": [f"D_TRA2_B7_{rank}" for rank in (1, 2, 3)],
    "D_TRA3_B7": [f"D_TRA3_B7_{rank}" for rank in (1, 2, 3)],
    "D_TRA4_B7": [f"D_TRA4_B7_{rank}" for rank in (1, 2, 3)],
    "D_TRA5_B7": [f"D_TRA5_B7_{rank}" for rank in (1, 2, 3)],
    "D_TRA6_B7": [f"D_TRA6_B7_{rank}" for rank in (1, 2, 3)],
}
RANK_WEIGHTS = {1: 3.0, 2: 2.0, 3: 1.0}

for rank_columns in RANK_VARIABLE_GROUPS.values():
    assert set(rank_columns) <= set(data.columns), (
        f"{rank_columns} 순위 열이 없습니다."
    )


def raw_codebook_key(column_name: str) -> str:
    """통합 열 이름을 원본 입력 코드북의 열 이름으로 되돌린다.

    `D_TRA_*`는 전처리 1)에서 `D_TRA1_*`~`D_TRA6_*`를 통합한
    열이라 원본 입력 코드북에는 `D_TRA1_*` 이름으로만 존재한다.

    Args:
        column_name: 통합 데이터프레임의 열 이름.

    Returns:
        원본 입력 코드북에서 조회 가능한 열 이름.
    """
    if column_name.startswith("D_TRA_"):
        return f"D_TRA1_{column_name[len('D_TRA_'):]}"
    return column_name


columns_before_rankw = data.columns.tolist()
rank_weighted_columns = {}
rank_weighted_metadata = {}
for prefix, rank_columns in RANK_VARIABLE_GROUPS.items():
    first_rank_column = rank_columns[0]
    codebook_key = raw_codebook_key(first_rank_column)
    value_labels = json.loads(
        codebook_by_name.loc[codebook_key, "integrated_value_labels"]
    )
    base_label = re.sub(
        r"-\d순위$",
        "",
        codebook_by_name.loc[codebook_key, "column_label"],
    )
    valid_mask = data[first_rank_column].notna().to_numpy()

    observed_codes = sorted(
        {
            raw_value
            for column in rank_columns
            for raw_value in data[column].dropna().unique()
        },
        key=float,
    )
    for raw_code in observed_codes:
        code_suffix = str(int(float(raw_code)))
        score = np.where(valid_mask, 0.0, np.nan)
        for rank, column in enumerate(rank_columns, start=1):
            matched = (data[column] == raw_code).to_numpy()
            score = np.where(
                matched, np.maximum(score, RANK_WEIGHTS[rank]), score
            )

        new_column = f"{prefix}_RANKW_{code_suffix}"
        rank_weighted_columns[new_column] = score
        rank_weighted_metadata[new_column] = {
            "prefix": prefix,
            "code": code_suffix,
            "category_label": value_labels[code_suffix],
            "base_label": base_label,
            "source_columns": rank_columns,
            "codebook_key": codebook_key,
        }

data = data.assign(**rank_weighted_columns)
print(f"순위 가중 다중응답 열 {len(rank_weighted_columns)}개 생성")

순위 가중 다중응답 열 95개 생성


In [19]:
new_column_order = columns_before_rankw.copy()
for prefix, rank_columns in RANK_VARIABLE_GROUPS.items():
    insert_at = new_column_order.index(rank_columns[-1]) + 1
    group_new_columns = [
        column
        for column in rank_weighted_columns
        if column.startswith(f"{prefix}_RANKW_")
    ]
    new_column_order = (
        new_column_order[:insert_at]
        + group_new_columns
        + new_column_order[insert_at:]
    )
data = data[new_column_order]

for prefix, rank_columns in RANK_VARIABLE_GROUPS.items():
    group_columns = [
        column for column in data.columns
        if column.startswith(f"{prefix}_RANKW_")
    ]
    expected_missing = data[rank_columns[0]].isna().sum()
    actual_missing = data[group_columns[0]].isna().sum()
    assert actual_missing == expected_missing, (
        f"{prefix} 그룹의 결측 처리가 1순위 결측과 일치하지 않습니다."
    )
    observed_scores = data[group_columns].stack().dropna()
    assert observed_scores.isin([0.0, 1.0, 2.0, 3.0]).all(), (
        f"{prefix} 그룹 가중 점수에 0~3 범위를 벗어난 값이 있습니다."
    )

print(f"순위 가중 변수 반영 후 크기: {data.shape[0]:,}행 × {data.shape[1]:,}열")

순위 가중 변수 반영 후 크기: 156,050행 × 1,476열


## 데이터 미리보기와 검증

In [21]:
print(f"최종 크기: {data.shape[0]:,}행 × {data.shape[1]:,}열")
preview_columns = [
    "YEAR", "ID", "D_TRA1_CASE", "D_TRA1_1_SPOT", "DQ3A",
    "A5_1", "A5_RANKW_1", "BSEX", "WT_DOM",
]
display(data[preview_columns].head())  # noqa: F821  # 결과 열 확인용 출력

최종 크기: 156,050행 × 1,476열


,YEAR,ID,D_TRA1_CASE,D_TRA1_1_SPOT,DQ3A,A5_1,A5_RANKW_1,BSEX,WT_DOM
0,2023,11010560931_124820,NaN,NaN,2.0,NaN,NaN,2.0,26559.50248115295
1,2023,11010560931_124821,1.0,37330.0,1.0,8.0,0.0,2.0,33746.01717435334
2,2023,11010560931_124823,NaN,NaN,0.0,NaN,NaN,1.0,15625.084604657755
3,2023,11010560931_124825,NaN,NaN,0.0,NaN,NaN,2.0,15620.054419226955
4,2023,11010560931_124833,1.0,32010.0,NaN,7.0,0.0,1.0,7209.363278762187


## 코드북 갱신과 저장

기존 코드북에서 삭제한 열의 행을 제거하고, `D_TRA_*`와 `DQ3A`는 새
행으로 추가해 최종 데이터 열과 1:1 대응하는 코드북을 만든다.

In [22]:
codebook_rows = []
for column_name in data.columns:
    if column_name in codebook_by_name.index:
        codebook_rows.append(codebook_by_name.loc[column_name].to_dict())
        continue

    # if column_name.startswith("D_TRA_") and column_name not in rank_weighted_metadata:
    #     suffix = column_name[len("D_TRA_"):]
    #     template = codebook_by_name.loc[f"D_TRA1_{suffix}"].to_dict()
    #     template["column_name"] = column_name
    #     template["comparison_status"] = "derived_with_assumption"
    #     template["notes"] = (
    #         "D_TRA1_..D_TRA6_ 중 CHECK == 'Y'인 여행 슬롯 값을 선택해 "
    #         "통합; 대상 여행이 없는 응답자는 결측"
    #     )
    #     for year in (2023, 2024, 2025):
    #         template[f"y{year}_source_name"] = (
    #             f"D_TRA1_{suffix}~D_TRA6_{suffix} 중 CHECK 슬롯 선택"
    #         )
    #     codebook_rows.append(template)
    #     continue

    if column_name == "DQ3A":
        template = codebook_by_name.loc["DQ3A_2"].to_dict()
        template["column_name"] = "DQ3A"
        template["column_label"] = "DQ3-1. 동거 손자/자녀 수 합계(명)"
        template["column_type"] = "scale"
        template["comparison_status"] = "derived_with_assumption"
        template["integrated_value_labels"] = np.nan
        template["notes"] = (
            "DQ3A_2(미취학)+DQ3A_3(취학)+DQ3A_4(성인) 손자/자녀 수 합; "
            "DQ3A_1('동거 손자/자녀 없음')은 0 검증에만 사용하고 "
            "합산에는 포함하지 않음"
        )
        for year in (2023, 2024, 2025):
            template[f"y{year}_source_name"] = "DQ3A_2+DQ3A_3+DQ3A_4"
        codebook_rows.append(template)
        continue

    if column_name in rank_weighted_metadata:
        meta = rank_weighted_metadata[column_name]
        template = codebook_by_name.loc[meta["codebook_key"]].to_dict()
        template["column_name"] = column_name
        template["column_label"] = (
            f"{meta['base_label']} 가중점수: {meta['category_label']}"
        )
        template["column_type"] = "scale"
        template["comparison_status"] = "derived_with_assumption"
        template["integrated_value_labels"] = np.nan
        template["notes"] = (
            f"{'/'.join(meta['source_columns'])}에서 카테고리 코드 "
            f"{meta['code']}({meta['category_label']})의 선택 순위에 "
            "가중치 부여; 1순위=3점, 2순위=2점, 3순위=1점, 미선택=0점, "
            "동일 카테고리가 여러 순위에 나오면 최고 순위 점수만 인정, "
            "1순위 문항이 결측인 응답자는 구조적 비해당으로 결측 유지"
        )
        for year in (2023, 2024, 2025):
            template[f"y{year}_source_name"] = (
                f"{'/'.join(meta['source_columns'])} 순위 가중치 변환"
            )
        codebook_rows.append(template)
        continue

    raise KeyError(f"{column_name}에 대응하는 코드북 행이 없습니다.")

new_codebook = pd.DataFrame(codebook_rows).reset_index(drop=True)
new_codebook["column_order"] = range(1, len(new_codebook) + 1)
assert list(new_codebook["column_name"]) == list(data.columns)
assert new_codebook["column_name"].is_unique

NATIONAL_TRAVEL_SURVEY_PREPROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
data.to_csv(OUTPUT_DATA_PATH, index=False, encoding="utf-8-sig")
new_codebook.to_csv(OUTPUT_CODEBOOK_PATH, index=False, encoding="utf-8-sig")
print(f"전처리 데이터 저장: {OUTPUT_DATA_PATH}")
print(f"전처리 코드북 저장: {OUTPUT_CODEBOOK_PATH}")

전처리 데이터 저장: C:\Users\ilove\Desktop\tourism_poster\data\preprocess\national_travel_survey\national_travel_survey_2023_2025_preprocessed_all_tour.csv
전처리 코드북 저장: C:\Users\ilove\Desktop\tourism_poster\data\preprocess\national_travel_survey\national_travel_survey_2023_2025_preprocessed_all_tour_codebook.csv


## 핵심 결과와 한계

- `D_TRA{k}_CHECK == "Y"` 슬롯을 응답자당 최대 1개로 확인하고, 그
  슬롯의 값을 `D_TRA_*` 열로 통합했다. A계열 본설문 응답 대상 여행이
  없는 응답자는 `D_TRA_*`가 결측이며, 이는 여행 경험이 없다는 뜻이
  아니라 A계열 문항에 응답하지 않았다는 구조적 비해당이다. 이 응답자
  비율은 위 품질 확인 셀의 `y_slot_count` 분포로 확인한다.
- 중복 질문 열과 근로 관련 불필요 열을 삭제했고, 손자녀 수 열
  `DQ3A_1~4`는 자녀 수 열 `DQ3A` 하나로 합쳤다.
- 순위형 변수 5그룹(`A3, A5, A5A, ZQ1, D_TRA_B7`)을 카테고리 코드별
  가중 다중응답 열(`{그룹}_RANKW_{코드}`)로 변환했다. 1순위 3점,
  2순위 2점, 3순위 1점, 미선택 0점을 부여했고, `A5A`·`ZQ1`처럼 같은
  카테고리가 여러 순위에 중복 응답된 경우 최고 순위 점수만
  인정했다. 그룹의 1순위 문항이 결측인 응답자는 구조적 비해당으로
  보고 결측을 유지했으며, 모델링 단계에서 모델별 결측 처리 전략을
  별도로 정해야 한다. 원본 순위 열은 삭제하지 않았다.
- 전체 열을 문자열로 읽어 명목형 코드가 숫자로 잘못 추론되지 않도록
  했다. `DQ3A`처럼 실제로 합산이 필요한 값만 명시적으로 숫자로
  변환했으므로, 이후 모델링 노트북에서도 각 열의 `column_type`을
  코드북에서 다시 확인한 뒤 사용해야 한다.
- `WT_DOM`은 응답자 수준 모집단 추정용 가중치이며, 이 노트북은 열
  구성만 정리했을 뿐 가중 통계를 산출하지 않았다.
- 결과는 `data/preprocess/national_travel_survey_2023_2025_preprocessed.csv`와
  짝이 되는 코드북에 저장했다. 모델별 최종 변수셋(분류·클러스터링)은
  이 파일에서 추가로 선택한다.

In [ ]:
import pandas as pd

codebook = pd.read_csv(
    r"C:\Users\ilove\Desktop\tourism_poster\data\preprocess\national_travel_survey\national_travel_survey_2023_2025_all_tour_codebook.csv", 
    encoding="utf-8-sig"
)

type_mapping = {
    "scale": "float64",
    "nominal": "string",
}

dtype_map = (
    codebook
    .dropna(subset=["column_name", "column_type"])
    .assign(
        pandas_dtype=lambda x: x["column_type"].map(type_mapping)
    )
    .dropna(subset=["pandas_dtype"])
    .set_index("column_name")["pandas_dtype"]
    .to_dict()
)

df = pd.read_csv(
    r"C:\Users\ilove\Desktop\tourism_poster\data\preprocess\national_travel_survey\national_travel_survey_2023_2025_all_tour.csv",
    encoding="utf-8-sig",
    dtype=dtype_map,
)

df.head()

,YEAR,ID,SA1_1,SA1_2,SA1_3,SA1_4,SA1_5,D_TRA1_CHECK,D_TRA1_CASE,D_TRA1_SYEAR,...,BAGE,BJOB,BINC1,BINC2,BEDU,BMAR,BFAM,BMON,BARA,WT_DOM
0,2023,11010560931_124820,0.0,0.0,0.0,0.0,0.0,<NA>,<NA>,NaN,...,5.0,11.0,6.0,1.0,4.0,2.0,3.0,1.0,1.0,26559.502481
1,2023,11010560931_124821,1.0,0.0,0.0,0.0,0.0,Y,1.0,2023.0,...,6.0,11.0,7.0,1.0,4.0,2.0,3.0,1.0,1.0,33746.017174
2,2023,11010560931_124823,0.0,0.0,0.0,0.0,0.0,<NA>,<NA>,NaN,...,7.0,13.0,2.0,1.0,3.0,2.0,2.0,1.0,1.0,15625.084605
3,2023,11010560931_124825,0.0,0.0,0.0,0.0,0.0,<NA>,<NA>,NaN,...,7.0,11.0,1.0,1.0,3.0,3.0,1.0,1.0,1.0,15620.054419
4,2023,11010560931_124833,1.0,0.0,0.0,0.0,0.0,Y,1.0,2023.0,...,2.0,1.0,5.0,5.0,4.0,1.0,1.0,1.0,1.0,7209.363279


In [26]:
import re

spot_cols = [
    col
    for col in df.columns
    if re.fullmatch(r"D_TRA\d+_\d+_SPOT", col)
]

In [29]:
import re

trip_spot_cols = {
    trip_no: sorted(
        [
            col
            for col in df.columns
            if re.fullmatch(rf"D_TRA{trip_no}_\d+_SPOT", col)
        ],
        key=lambda col: int(re.search(r"_(\d+)_SPOT$", col).group(1)),
    )
    for trip_no in range(1, 7)
}


def count_region_changes(row: pd.Series) -> int:
    """연속으로 중복된 지역을 하나의 방문 구간으로 계산합니다."""
    spots = row.dropna()

    if spots.empty:
        return 0

    return spots.ne(spots.shift()).sum()


for trip_no, cols in trip_spot_cols.items():
    df[f"D_TRA{trip_no}_REGION_COUNT"] = (
        df[cols]
        .apply(count_region_changes, axis=1)
    )

C:\Users\ilove\AppData\Local\Temp\ipykernel_25476\3206016320.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"D_TRA{trip_no}_REGION_COUNT"] = (
C:\Users\ilove\AppData\Local\Temp\ipykernel_25476\3206016320.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"D_TRA{trip_no}_REGION_COUNT"] = (
C:\Users\ilove\AppData\Local\Temp\ipykernel_25476\3206016320.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider join

In [30]:
region_count_cols = [
    f"D_TRA{trip_no}_REGION_COUNT"
    for trip_no in range(1, 7)
]

region_count_distribution = (
    df[region_count_cols]
    .stack()
    .loc[lambda x: x > 0]
    .value_counts()
    .sort_index()
    .rename_axis("방문 지역 구간 수")
    .reset_index(name="여행 수")
)

region_count_distribution["비율"] = (
    region_count_distribution["여행 수"]
    / region_count_distribution["여행 수"].sum()
)

region_count_distribution

,방문 지역 구간 수,여행 수,비율
0,1,75813,0.895362
1,2,6448,0.076152
2,3,1782,0.021046
3,4,416,0.004913
4,5,157,0.001854
5,6,36,0.000425
6,7,15,0.000177
7,8,6,0.000071


In [12]:
df.head().to_csv(
    r"C:\Users\ilove\Desktop\tourism_poster\data\preprocess\national_travel_survey\national_travel_survey_2023_2025_preprocessed_head.csv",
    index=False,
    encoding="utf-8-sig",
)